# Финальная модель: посылка на официальном тесте

Выберите GPU **T4**, загрузите `final_checkpoint360000_colab.zip` через панель Files и выполните ячейки сверху вниз. Архив содержит проверенные финальные веса checkpoint-360000, BPE, код инференса и 1919 официальных вопросов в исходном порядке.

Beam 4, batch 128, FP16 autocast, лимит 256 новых токенов с обычной остановкой по EOS. Обучение и выполнение сгенерированного кода не запускаются. Время генерации включает токенизацию и декодирование; установка библиотек и загрузка весов выполняются заранее. Последняя ячейка выдаёт JSONL только с полем `code` для самостоятельной отправки боту.

In [ ]:
%pip -q install "transformers==5.10.2" "tokenizers==0.22.2" "datasets==4.6.0" safetensors

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys
import time
import zipfile

archive = Path('/content/final_checkpoint360000_colab.zip')
expected_bytes = 518363693
expected_sha256 = 'dd822e91abb618cb79b99b1dc900a19c6df9f3e91d52671cd21c85214947ae69'
for attempt in range(240):
    size = archive.stat().st_size if archive.exists() else 0
    if size == expected_bytes:
        break
    if size > expected_bytes:
        raise ValueError('Unexpected archive size')
    if attempt % 6 == 0:
        print(f'Upload: {size / expected_bytes:.1%} ({size:,}/{expected_bytes:,} bytes)', flush=True)
    time.sleep(5)
else:
    raise TimeoutError('Upload is incomplete; rerun this cell after upload finishes')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

assert sha256_file(archive) == expected_sha256, 'Archive SHA-256 mismatch'
project = Path('/content/nlp_hw2_final360000')
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    bundle.extractall(project)
manifest = json.loads((project / 'manifest.json').read_text())
for name, info in manifest['files'].items():
    assert sha256_file(project / name) == info['sha256'], name
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

import torch
import datasets
import transformers
import tokenizers
from inference import InferenceConfig, generate_code, load_inference_model

assert transformers.__version__ == '5.10.2'
assert tokenizers.__version__ == '0.22.2'
assert datasets.__version__ == '4.6.0'
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
torch.set_num_threads(2)
torch.manual_seed(42)
final_test = datasets.Dataset.from_parquet(str(project / 'test.parquet'))
questions = list(final_test['question'])
assert len(questions) == manifest['test_rows'] == 1919
question_hash = hashlib.sha256(json.dumps(questions, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
assert question_hash == manifest['question_order_sha256']
final_model, final_tokenizer = load_inference_model(project / 'checkpoint', device='cuda:0')
assert final_model.count_parameters() == manifest['model_parameters'] == 139253760
assert final_model.network.generation_config.min_new_tokens in (None, 0)
assert max(len(item.ids) for item in final_tokenizer.encode_batch(questions)) <= 256
print('READY: checkpoint-360000; 1919 questions in original order; all SHA-256 verified.', flush=True)
print('GPU:', torch.cuda.get_device_name(0), '| attention:', final_model.network.config._attn_implementation)
print('Versions:', torch.__version__, transformers.__version__, tokenizers.__version__, datasets.__version__)
print('Architecture:', final_model.config.encoder_layers, '+', final_model.config.decoder_layers,
      '| hidden:', final_model.config.d_model, '| parameters:', final_model.count_parameters())


In [ ]:
# Full official test, one deterministic beam-search answer per question.
import base64
from IPython.display import HTML, display

final_config = InferenceConfig(
    tokenizer=final_tokenizer, device='cuda:0', mixed_precision='fp16',
    batch_size=128, max_source_length=256, max_new_tokens=256,
    num_beams=4, sort_by_length=True, truncate_source=False, output_dir=None,
)
print('START: checkpoint-360000 / beam 4 / batch 128 / FP16 / max_new_tokens 256', flush=True)
torch.cuda.reset_peak_memory_stats()
final_result = generate_code(final_test, final_model, final_config)
assert list(final_result.predictions['question']) == questions
final_codes = final_result.codes
assert len(final_codes) == 1919 and all(isinstance(code, str) and code.strip() for code in final_codes)
final_submission = Path('/content/transformer_002_checkpoint360000_beam4_t4.jsonl')
final_submission.write_text(''.join(json.dumps({'code': code}, ensure_ascii=False) + '\n' for code in final_codes), encoding='utf-8')
final_rows = [json.loads(line) for line in final_submission.read_text(encoding='utf-8').splitlines()]
assert len(final_rows) == 1919 and all(set(row) == {'code'} for row in final_rows)
assert [row['code'] for row in final_rows] == final_codes
seconds = final_result.metrics['total_seconds']
print('DONE:', final_submission.name)
print('Rows: 1919 | original order: verified | fields: code only')
print('Bytes:', final_submission.stat().st_size, '| SHA256:', sha256_file(final_submission))
print('Full generation seconds (including tokenization/decoding):', round(seconds, 3))
print('Within 180 seconds:', seconds <= 180)
print('Peak allocated VRAM GiB:', round(torch.cuda.max_memory_allocated() / 2**30, 3))
print('Mean tokens:', round(final_result.metrics['mean_generated_tokens'], 3))
print('Hit token limit:', sum(final_result.predictions['hit_token_limit']))
if seconds <= 180:
    encoded = base64.b64encode(final_submission.read_bytes()).decode('ascii')
    display(HTML(f'<a download="{final_submission.name}" href="data:application/octet-stream;base64,{encoded}">Скачать финальный JSONL — 1919 ответов</a>'))
else:
    print('Time limit exceeded: tune inference and regenerate before submitting.')
